# LoRA fine-tuning on a free GPU

## Teaching a 3-billion-parameter model to read 1930s German handwriting, on Google Colab

[Open this notebook in Colab](https://colab.research.google.com/github/carpentries-incubator/ML_with_AWS_SageMaker/blob/main/finetune_extras/colab/LoRA-finetuning-on-a-free-GPU.ipynb)

This notebook fine-tunes CHURRO-3B, a vision-language model, to transcribe handwritten German letters from the 1930s, on the free GPU that Google Colab provides. It costs nothing, and the whole run, with the model scored before and after, takes about an hour and a half.

It is the fine-tuning episode of a [Carpentries lesson on machine learning on AWS](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/08_Finetuning-BadgerScribe.html), with the cloud account taken out. The data, the model and the training script are the same. The GPU is smaller, 16 GB with no bf16 support, so the frozen weights are stored in 4-bit. The section on what fits a free GPU explains that.

Before running anything, give the notebook a GPU. In Colab's menu choose **Runtime → Change runtime type**, pick **T4 GPU**, and save. You also need a Google account, for Colab and for Google Drive, where the training checkpoints go so that a dropped connection costs one epoch and not the run.

By the end you will have fine-tuned a vision-language model, compared it with the original on the same held-out pages, read the learning curve to decide how many passes were worth it, and, if you want, transcribed the pages nobody has labeled.

## Three places to run this fine-tune

This notebook is the Colab column. The same fine-tune, with the same model, data and scripts, is the [lesson's episode](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/08_Finetuning-BadgerScribe.html) on AWS and a [guide for UW-Madison's CHTC](https://github.com/carpentries-incubator/ML_with_AWS_SageMaker/tree/main/finetune_extras/chtc).

| | [AWS SageMaker](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/08_Finetuning-BadgerScribe.html) | [Google Colab](https://colab.research.google.com/github/carpentries-incubator/ML_with_AWS_SageMaker/blob/main/finetune_extras/colab/LoRA-finetuning-on-a-free-GPU.ipynb) | [CHTC](https://github.com/carpentries-incubator/ML_with_AWS_SageMaker/tree/main/finetune_extras/chtc) |
|---|---|---|---|
| Who can use it | Anyone with an AWS account and a card or credits | Anyone with a Google account | UW-Madison researchers and students, with a faculty sponsor |
| Cost of this run | About `$0.75` | Free | Free |
| GPU | A10G, 24 GB, bf16 | T4, 16 GB, no bf16, so the frozen weights go to 4-bit | L40S, A100, H100 or H200, 48 to 141 GB, bf16; NVLink on the A100, H100 and H200 servers |
| Time for this run | 33 minutes, starts in a minute | About an hour and a half, starts at once if a GPU is free | About 20 minutes, after a queue wait of minutes to hours |
| Largest LoRA fine-tune | No hard limit, only cost. 7B on the lesson's `$1.41`/h card, 70B at `$13`/h | about 7B, in 4-bit | about 50B in bf16 on an H200, a 70B in 4-bit on one 80 GB card, larger across a server's cards |
| Largest full fine-tune | No hard limit, only cost. 3B at `$7`/h, 70B at `$127`/h | none, a 3B needs 52 GB | a 7B on one H200, a 70B across the H100 or H200 servers' cards |
| What you learn beyond the fine-tune | Estimating and reading a bill, tags, job guard rails | Fitting a model into a small GPU | Describing a job for a scheduler, containers, staging data |

For fine-tuning, CHTC is the first choice for UW-Madison researchers. It is free, its GPUs are large, and the queue is the only price. This notebook is for practicing LoRA with nothing to set up. The cloud has the newest and fastest GPUs, and a fine-tune on a large dataset sometimes needs them, but it bills by the hour, so it is the choice when the other two cannot do the job, not the default.

None of the three runtime limits is a wall. The training script writes a checkpoint at the end of every epoch and resumes from it, so a run longer than Colab's session, CHTC's job lane or an AWS runtime cap is several runs that each pick up where the last one stopped. Here the checkpoints go to Google Drive, which is what the Drive step below is for.

> **Colab is a place to study fine-tuning, not to do it for a project.** The free tier gives a GPU when one is free, ends a session after a few hours of use or 90 minutes idle, and cuts back after heavy days. That is enough for this notebook. It is a poor fit for a research project, where runs repeat, data grows and results have to be reproducible months later. At UW-Madison, take what this notebook teaches to CHTC, or to the cloud when CHTC cannot do the job.

> **A larger dataset can outrun the session.** The script saves a checkpoint at the end of each epoch, and nothing in between. This run's epoch is 251 pages, about 30 minutes on a T4. With ten times the pages, one epoch is longer than the session, so the session ends before the first checkpoint is written and the next session starts from nothing. If you reuse this notebook on your own data, keep an epoch under about two hours with `--max_train_pages`, or move the job to CHTC, where a job lane is 12 hours or more.


## Why fine-tune a model?

Small models can be trained from scratch. That works when the problem is well defined and there is enough training data, and it is not practical for tasks that need the broad capabilities of a modern language or vision model with billions of weights. Training one of those from scratch takes thousands of GPUs running for months, tens of millions of GPU-hours, and enormous datasets. One pass of a 7-billion-parameter language model over a million papers is about four days on a four-GPU machine, and the models behind ChatGPT are far larger and trained on far more.

Fortunately, we don't have to start from scratch. ***Pretrained*** models already know how to interpret images, understand language, and generate text. ***Fine-tuning*** lets us build on those capabilities by training a model further on examples from our own task.

Consider historical handwriting recognition. A general-purpose vision-language model, such as the open-weight Qwen3.8-27B or a commercial one like Gemini, may be able to transcribe ordinary printed text and modern handwriting, yet struggle with nineteenth-century documents, unfamiliar letterforms, or the conventions used by a particular archive. Giving it better instructions can help, but there are limits to what prompting can accomplish when the model has never learned to recognize the writing itself.

Fine-tuning offers another approach. Show the model examples of the documents alongside their correct transcriptions, and train it to produce better results on similar material.

This can be surprisingly effective. [CHURRO](https://github.com/stanford-oval/Churro) is a 3-billion-parameter vision-language model developed for historical text recognition. Its researchers fine-tuned an existing Qwen model on roughly 100,000 historical document pages and reported better transcription accuracy than Gemini 2.5 Pro at substantially lower inference cost ([paper](https://arxiv.org/abs/2509.19768)). Instead of using an ever larger general-purpose model, they trained a smaller one to become particularly good at the task they cared about. The same pattern shows up in other fields.

![Three specialist models against much larger general ones, on each task's own benchmark](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/fig/specialists-vs-large-models.png)

*Three specialist models against much larger general ones, on each task's own benchmark. Sources: [CHURRO, EMNLP 2025](https://arxiv.org/abs/2509.19768); [Goedel-Prover-V2, ICLR 2026](https://arxiv.org/abs/2508.03613); [MedGemma technical report, 2025](https://arxiv.org/abs/2507.05201). Scores as reported by each model's authors.*

That is the approach here. We take CHURRO, which already specializes in historical documents, and adapt it further to a collection of handwritten German letters from UW-Madison's Badger Scribe challenge.

### How fine-tuning works

Fine-tuning uses the same basic training process as any model. The model makes predictions on training examples, we measure how far those predictions are from the correct answers, and gradient descent adjusts its parameters to reduce the error.

The difference is where we begin. Instead of initializing the model with random weights, we start with weights learned during an earlier, much larger training process. Our examples don't need to teach the model everything about language or images. They only need to help it adapt what it already knows.

We could update every parameter in the model, which is called ***full fine-tuning***. That is how CHURRO itself was created. But even a relatively small model with 3 billion parameters is expensive to train this way, because training needs far more GPU memory than running the model does, and on the free GPU this notebook uses it is out of the question. The next section says why.

### LoRA adds new small weights beside the frozen model

The alternative to full fine-tuning is ***parameter-efficient fine-tuning***, or ***PEFT***, a family of methods that leave the pretrained weights frozen and train a small number of new parameters instead. ***Low-rank adaptation***, or ***LoRA***, is the most widely used form of PEFT. It keeps the pretrained weights frozen and adds small sets of trainable parameters to selected layers. Instead of directly changing a large weight matrix, LoRA learns a much smaller adjustment to its output.

![Full fine-tuning updates the whole weight matrix; LoRA freezes it and trains two thin matrices beside it](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/fig/lora.svg)

*Full fine-tuning updates the whole weight matrix; LoRA freezes it and trains two thin matrices beside it.*

Nothing in the original model is trained. The trained parameters are the new matrices, and they are a small fraction of the model's size, under one percent here. We still run the whole model during training, but we only calculate and store weight updates for the new matrices.

LoRA also produces a relatively small ***adapter***, about 100 MB, that we save separately from the original model. To use the fine-tuned model later, we load the pretrained weights together with the adapter.

This makes LoRA particularly useful for research groups. We can take an existing open-weight model, adapt it to a specialized dataset, evaluate whether the adaptation helped, and keep the resulting model without the expense of full fine-tuning.

### Training, full fine-tuning and LoRA on one scale

The GPU-hours put the three on one scale. Training from scratch is at the top, the full fine-tune that made CHURRO in the middle, and this notebook's LoRA run at the bottom.

![GPU-hours, on a log scale, to train a 3B model from scratch, to fully fine-tune it, and to LoRA fine-tune it](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/fig/pretrain-finetune-lora.svg)

*GPU-hours, on a log scale, to train a 3B model from scratch, to fully fine-tune it, and to LoRA fine-tune it.*

Why do the three differ so much? Training from scratch and full fine-tuning do the same work per step. Every parameter is updated, which means keeping four things in GPU memory for each one, the weight itself, its gradient, a 32-bit copy of the weight for the update to land in, and the optimizer's two running averages. That is 16 bytes per parameter, so a 3B model needs about 60 GB before activations and other overhead, which means several GPUs. What separates those two is how much data they go through, trillions of tokens against a hundred thousand pages. LoRA cuts the work in a different place. It updates only the adapter and leaves the model's own weights frozen, and a frozen weight costs 2 bytes, not 16, with nothing else to store for it.

### What fits a free GPU

Colab's free GPU is a T4 with 16 GB of memory. The table below is the memory arithmetic for CHURRO's 3.75 billion parameters on it.

![GPU memory per parameter: inference, a LoRA fine-tune, and a full fine-tune](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/fig/memory-per-parameter.svg)

*GPU memory per parameter: inference, a LoRA fine-tune, and a full fine-tune.*

| How the 3.75B parameters are used | Bytes per parameter | GPU memory with activations | Fits a 16 GB T4? |
|---|---|---|---|
| **Full fine-tune**, every weight updated | 16 | about 60 GB | No. That is a four-GPU machine. |
| **LoRA**, weights frozen in 16-bit, the adapter trained | 2, plus the adapter | about 12 GB | Only just, and the T4 cannot compute in bf16, the 16-bit format newer GPUs train in. |
| **QLoRA**, weights frozen in 4-bit, the same adapter | 0.5, plus the adapter | about 6 GB | Yes, with room. |

***QLoRA*** is LoRA with the frozen weights stored in 4-bit. Nothing about what the model learns changes. The adapter is the same, the arithmetic is the same, and the frozen weights are read from a compressed copy. On a 24 GB or larger GPU the lesson runs plain LoRA in bf16. On the T4 we store the weights in 4-bit and compute in fp16, the older 16-bit format the T4 does support, and the training script picks both up from one flag.

### Our task, transcribing historical German letters

The data is from the [Badger Scribe](https://www.kaggle.com/competitions/badger-scribe) challenge, which asks participants to produce faithful transcriptions of scanned historical documents. The competition includes several kinds of archival material. We concentrate on letters from the Anna Seifert correspondence, held by the Max Kade Institute, 191 letters and postcards written between 1932 and 1942 by one woman to her son, transcribed by the Institute's Kurrent Transcription Group.

These letters were written in ***Kurrent***, a historical form of German handwriting whose letterforms differ substantially from modern cursive. Even a model trained on historical documents can struggle with it, which makes this a useful test of whether additional fine-tuning helps.

We have 295 labeled pages from this collection. The training script holds about 15% of them out for evaluation and trains on the remaining 251, with the human transcriptions as targets.

| split | pages | documents | label source |
|---|---|---|---|
| train | 295 | 4 letter bundles | all human-transcribed, ~620 characters per page |
| test | 141 | 5 other letter bundles | labels held by Kaggle |

The question is simple. Does fine-tuning CHURRO on these examples improve its transcription of pages it has never seen? We measure that with ***character error rate (CER)***, the competition's own metric, edits per reference character. Lower is better.

### What we'll do

In order: check the GPU and install three libraries, download the data and look at a page, measure how many tokens a page is because that sets the training time, work out how many parameters LoRA will train from the model's own config file, run a two-minute smoke test and then the real fine-tune with checkpoints on Google Drive, compare the model before and after on the same held-out pages and read the learning curve, and optionally transcribe the 141 unlabeled test pages.

## Setup

### The GPU

`nvidia-smi` names the card and its memory. The second line prints the GPU's compute capability, which decides the number formats. bf16, the 16-bit format newer GPUs train in, needs capability 8.0 or newer. A T4 is 7.5, so on it the frozen base weights are stored in 4-bit, as the table above said, and the adapter, the only part that trains, computes in fp16, the 16-bit format the T4 does support. The training script makes the same choice on its own from the `--dtype auto` flag.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
major, minor = torch.cuda.get_device_capability(0)
print(f"compute capability {major}.{minor}")
print("bf16 needs 8.0 or newer. On a T4 (7.5) the frozen base weights are stored in 4-bit and the adapter is trained in fp16.")

Install the three libraries the training script needs beyond what Colab ships. `transformers` loads the model, `peft` adds the LoRA adapter, and `bitsandbytes` stores the frozen weights in 4-bit. The pin on `transformers` keeps the version the lesson's training job was tested with.

In [ ]:
%pip install -q "transformers>=4.56,<5" "peft>=0.14" bitsandbytes accelerate   # the 4.x series the lesson's job was tested with; transformers 5 changed the Trainer's arguments
%pip uninstall -y -q torchao   # Colab ships an old torchao that peft refuses when it loads the adapter onto a 16-bit model, which the last step does


### Google Drive, for the checkpoints

Colab ends a session after about 12 hours, or after 90 minutes idle, and sometimes sooner. The training script saves a checkpoint at the end of every epoch. If those checkpoints live on Google Drive, a dropped session loses at most one epoch, and rerunning the training cell resumes from the last one.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
work = "/content/badger-scribe"                                   # data, code and outputs for this session; gone when the session ends
ckpt_dir = "/content/drive/MyDrive/badger-scribe-checkpoints"     # on Drive, so it survives a disconnect
os.makedirs(work, exist_ok=True); os.makedirs(ckpt_dir, exist_ok=True)
%cd {work}

### The code

The training script, the scoring code and a batch transcriber come from the lesson's repository. Only its `scripts/` folder is needed here. The training script is the one the lesson runs on AWS, unchanged, and we read it before running it, in the fine-tuning section.

In [ ]:
!rm -rf lesson && git clone -q --depth 1 https://github.com/carpentries-incubator/ML_with_AWS_SageMaker.git lesson   # a fresh copy every time, so a rerun gets the current scripts
!cp lesson/scripts/badger_common.py lesson/scripts/finetune_churro_lora.py lesson/scripts/churro_transcribe_batch.py .
!grep -c "load_in_4bit" finetune_churro_lora.py   # 5 or more means the script is the current one

### The data

The lesson publishes the Badger Scribe letters as a GitHub release, three zip files of about 130 MB in all: the labels and metadata, the training page images, and the test page images.

In [ ]:
release = "https://github.com/carpentries-incubator/ML_with_AWS_SageMaker/releases/download/badger-scribe-data-v1"
for part in ["labels", "images-train", "images-test"]:
    !curl -sSL -o {part}.zip {release}/badger-scribe-v1-{part}.zip && unzip -q -o {part}.zip -d data && rm {part}.zip
!ls data
!ls data/images | wc -l

`train.csv` has one row per labeled page, with the page's id, the letter bundle it belongs to, and the transcription. The test pages have no labels. Kaggle keeps those.

In [ ]:
import pandas as pd
train_df = pd.read_csv("data/train.csv", keep_default_na=False)   # keep_default_na=False: an empty transcription stays "" instead of becoming NaN
test_df = pd.read_csv("data/test.csv", keep_default_na=False)
print("labeled training pages:", len(train_df), " test pages:", len(test_df))
print(train_df.groupby("doc_id").size().rename("pages per letter bundle"))
train_df.head(3)

Look at one page and its transcription. Verbatim means the writer's spelling and punctuation, umlauts and ß as written, Kurrent letterforms mapped to their Latin equivalents, and `#` for a character the transcriber could not read.

In [ ]:
from IPython.display import Image, display
row = train_df.iloc[3]
print(row.page_id, "|", row.doc_id)
print(row.text[:600])
display(Image(filename=f"data/images/{row.page_id}.jpg", width=500))

## How many tokens is a page?

Training time is proportional to how many tokens the model processes, and a page image is tokens too. The model resizes the image to at most `max_pixels` and turns every 28 × 28 block of pixels into one token. The transcription becomes pieces of words, more pieces for German and for old spellings than for English. The count depends on the model's own tokenizer and image processor, so measure it with them. Loading the processor downloads a few megabytes of configuration, not the 7.5 GB of weights.

In [ ]:
from transformers import AutoProcessor
import badger_common as bc

processor = AutoProcessor.from_pretrained("stanford-oval/churro-3B", min_pixels=bc.DEFAULT_MIN_PIXELS, max_pixels=bc.DEFAULT_MAX_PIXELS)

image = bc.load_page_image(f"data/images/{row.page_id}.jpg", bc.DEFAULT_MAX_PIXELS)   # resized the way the training script resizes it
prompt = bc.build_prompt("kade_letters")
full = processor.apply_chat_template(bc.full_messages(prompt, row.text), tokenize=False)
enc = processor(text=[full], images=[image], return_tensors="pt")
n_total = enc["input_ids"].shape[1]
n_prompt = processor(text=[processor.apply_chat_template(bc.user_messages(prompt), tokenize=False, add_generation_prompt=True)],
                     images=[image], return_tensors="pt")["input_ids"].shape[1]
n_image = int((enc["image_grid_thw"].prod(dim=1) // 4).sum())   # 14-pixel patches merged 2x2: one token per 28x28 block

print(f"page {row.page_id}: {image.size[0]} x {image.size[1]} px after resizing -> {n_image} image tokens")
print(f"instruction prompt, the same on every page: {n_prompt - n_image} tokens")
print(f"transcription: {len(row.text.split())} words -> {n_total - n_prompt} tokens")
print(f"tokens the model sees for this page: {n_total}")

The image is close to the 1,280-token cap whatever the scan's original size, because every scan is larger than a megapixel and gets shrunk to it. The transcription comes out near two tokens per word, where English text is about 1.3. One page is a point, not an average, so count the text tokens of every page with the tokenizer alone, which takes seconds.

In [ ]:
text_tokens = train_df.text.map(lambda t: len(processor.tokenizer(t)["input_ids"]))
avg_tokens_per_page = n_image + (n_prompt - n_image) + text_tokens.mean()
print(f"text tokens per page: mean {text_tokens.mean():.0f}, min {text_tokens.min()}, max {text_tokens.max()}")
print(f"average tokens per page: about {avg_tokens_per_page:.0f}")
print(f"tokens per pass over the training pages: about {avg_tokens_per_page * len(train_df) * 0.85 / 1e6:.2f} million")   # 85%: the script holds 15% of pages out for scoring

That last number, times the number of passes, is the work the GPU does. Halve `max_pixels` and the image tokens halve with it, and so does the training time, at some cost in accuracy on small handwriting.

## How many parameters will train?

The adapter rank decides how many parameters LoRA trains, and that count is worth knowing before the run for two reasons. The script refuses to train if more than 5% of the model turns out to be trainable, because that would mean the base weights were not frozen, and the number appears in the training log as `trainable params`, so working it out now tells you whether the run is doing what you asked.

LoRA attaches to seven weight matrices in every layer of the language model, the four attention projections and the three in the MLP. An adapter of rank r on a matrix of shape in × out adds r × (in + out) parameters, the two thin matrices in the LoRA figure. Add that up over the seven, multiply by the number of layers, and you have the count. The widths come from the model's `config.json`, which for any model on Hugging Face is under **Files and versions** on the model page.

In [ ]:
import json
from huggingface_hub import hf_hub_download

with open(hf_hub_download("stanford-oval/churro-3B", "config.json")) as f:
    cfg = json.load(f)
text = cfg.get("text_config", cfg)            # newer configs keep the language model's sizes under text_config

layers = text["num_hidden_layers"]            # how many times the seven matrices repeat
hidden = text["hidden_size"]                  # the width of the model's main vector
inter = text["intermediate_size"]             # the MLP's wider middle layer
kv = hidden * text["num_key_value_heads"] // text["num_attention_heads"]   # k_proj and v_proj are narrower: several heads share one key and value

projections = {                               # (in, out) of each matrix LoRA attaches to, in one layer
    "q_proj": (hidden, hidden), "k_proj": (hidden, kv), "v_proj": (hidden, kv), "o_proj": (hidden, hidden),
    "gate_proj": (hidden, inter), "up_proj": (hidden, inter), "down_proj": (inter, hidden),
}
total_params = 3.75e9                         # the whole model, vision tower included
for lora_r in (8, 16, 32):
    trainable = sum(lora_r * (i + o) for i, o in projections.values()) * layers
    print(f"rank {lora_r:>2}: {trainable / 1e6:5.1f}M trainable parameters, {100 * trainable / total_params:.2f}% of the model")

Rank 16, used below, trains about 30 million parameters, under 1% of the model. Their gradients and optimizer state at 16 bytes each are about half a gigabyte, which is why the run fits beside the frozen weights.

## The fine-tune

### Read the script first

Open `finetune_churro_lora.py` in Colab's file browser (the folder icon on the left) before you run it. It is about 400 lines, and these are the parts to find. `split_pages` holds 15% of the pages out of training with a fixed seed, so every run scores the same pages. `LoraConfig` names the seven projections in its `target_modules` pattern and excludes the vision tower. Two safeguard checks right after `get_peft_model` stop the run if the adapter landed on the vision tower or if more than 5% of the model is trainable. And `evaluate` transcribes holdout pages and computes the CER.

### A smoke test

Every setting below reaches the script as a command-line flag. Before the real run, a short version, 20 pages for one pass with three pages scored, to catch a bad path or a missing library while it is cheap. It takes about seven minutes, two of them training. It also downloads the 7.5 GB of weights, once, into Colab's Hugging Face cache.

In [ ]:
!python finetune_churro_lora.py \
    --train_dir data --model_dir output/smoke --output_data_dir output/smoke-data --checkpoint_dir output/smoke-ckpt \
    --dtype auto --load_in_4bit 1 \
    --max_train_pages 20 --epochs 1 --eval_pages 3 --curve_pages 0 --grad_accum 4

Look for four lines in the output. `pages: 295 total -> 20 train / 44 holdout`, the split, with the 20 training pages this test asked for and the 44 holdout pages every run scores. `loading stanford-oval/churro-3B in fp16, frozen weights in 4-bit`, the precision chosen for this GPU. `trainable params: ...`, which should be about 30 million, the number from the section above. And `safeguards: adapter is ... (0.80%), vision tower frozen`, the two checks passing.

### The real run

The same script on all 251 training pages for two passes, scoring 15 holdout pages before and after and 8 of them at the end of each pass. On a T4 the training takes about an hour, with the two end-of-pass scorings inside it, and the scoring before and after about half an hour, so an hour and a half in all. The base model is slower to score than the fine-tuned one, 67 seconds a page against 38, because on a page it cannot read it keeps generating until the token limit. The checkpoints go to Drive. If Colab disconnects, run this cell again with the same `run_name` and it resumes from the last completed epoch.

In [ ]:
run_name = "churro-lora-2ep"          # change it to start a fresh run instead of resuming one
os.makedirs(f"{ckpt_dir}/{run_name}", exist_ok=True)   # the script writes checkpoints here only if the folder exists

!python finetune_churro_lora.py \
    --train_dir data --model_dir output/{run_name} --output_data_dir output/{run_name}-data --checkpoint_dir {ckpt_dir}/{run_name} \
    --dtype auto --load_in_4bit 1 \
    --epochs 2 --lr 2e-4 --lora_r 16 --batch_size 1 --grad_accum 4 \
    --eval_pages 15 --eval_before 1 --curve_pages 8

While it runs, the log shows the base model's CER on the holdout pages first, then the Trainer's loss every five steps, then a CER point at the end of each epoch, then the fine-tuned model's CER on the same pages.

## What changed

The script wrote everything into the output folder: the adapter, the per-page scores before and after, the curves and a summary.

In [ ]:
out = f"output/{run_name}"
print(sorted(os.listdir(out)))
summary = json.load(open(f"{out}/train_summary.json"))
print(json.dumps(summary, indent=2))

Both evaluations ran on the same 15 held-out pages, so the difference is the fine-tune and nothing else.

In [ ]:
before = pd.read_csv(f"{out}/eval_before.csv", keep_default_na=False)
after = pd.read_csv(f"{out}/eval_after.csv", keep_default_na=False)

comparison = before[["page_id", "cer"]].merge(after[["page_id", "cer"]], on="page_id", suffixes=("_base", "_lora"))
comparison["gain"] = comparison.cer_base - comparison.cer_lora   # positive: the fine-tune helped on that page
print(comparison.sort_values("gain", ascending=False).round(3).to_string(index=False))
print(f"\nmean CER  base: {comparison.cer_base.mean():.3f}   fine-tuned: {comparison.cer_lora.mean():.3f}")

Look at the page that improved most. The fine-tuned model should have picked up the conventions, umlauts and ß as written, no modernised spelling, Kurrent *s* and *e* read correctly, as well as this writer's hand.

In [ ]:
best = comparison.sort_values("gain", ascending=False).iloc[0].page_id
b, a = before.set_index("page_id"), after.set_index("page_id")
print("REFERENCE\n", b.loc[best, "text_ref"][:500])
print("\nBASE\n", b.loc[best, "text_pred"][:500])
print("\nFINE-TUNED\n", a.loc[best, "text_pred"][:500])
display(Image(filename=f"data/images/{best}.jpg", width=500))

### How many passes were worth it

Before and after says whether the fine-tune helped. The curve says whether two passes was the right amount. The script recorded the training loss every five steps and scored eight holdout pages at the end of each pass, with the base model as epoch 0.

In [ ]:
import matplotlib.pyplot as plt

log = json.load(open(f"{out}/train_log.json"))
loss = pd.DataFrame(log["loss"]); curve = pd.DataFrame(log["holdout_cer_by_epoch"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(loss.epoch, loss.loss, marker="."); ax1.set(xlabel="epoch", ylabel="training loss", title="Training loss")
ax2.plot(curve.epoch, curve.macro_cer, marker="o"); ax2.set(xlabel="epoch (0 = base model)", ylabel="holdout CER", title=f"Holdout CER on {log['curve_pages']} pages")
ax2.set_xticks(curve.epoch); plt.tight_layout(); plt.show()
print(curve.to_string(index=False))

Read the right-hand plot, not the left. Training loss almost always keeps falling, because it is measured on the pages the model is being fitted to. The holdout CER is the number that matters, and it has three shapes. Still falling at the last epoch means another pass would probably help. Flat means the data has given what it can. Rising while the loss still falls means the model is memorizing the training pages, so fewer passes or more data. Eight pages is a noisy estimate, so read the trend, not the third decimal.

To try a change, give the run a new name and relaunch the real run above. Four passes instead of two, a rank of 32, or `--max_pixels 2007040` (2,560 blocks, twice the resolution and twice the time), each changes one thing, and the before-and-after table says what it bought.

## Optional: transcribe the test pages

The batch transcriber uses the adapter to transcribe any list of pages. Here it writes a prediction for each of the 141 test pages, the ones nobody has labeled, which takes about an hour on a T4.

The result is not a Kaggle submission. This notebook's data is the German-letter part of [Badger Scribe](https://www.kaggle.com/competitions/badger-scribe), and the competition's test set has other document categories too, so a submission needs a row for every page in the competition's own `sample_submission.csv`. To get a leaderboard score, download the full competition data from Kaggle, train on all of its training pages, which takes longer than a Colab session and is a job for CHTC or the cloud, run this transcriber over all of its test pages, and build the file from its sample submission. Two submissions a day, and only that step needs a Kaggle account.

In [ ]:
pred_dir = "predictions"
os.makedirs(f"{pred_dir}/data", exist_ok=True)
test_df[["page_id", "doc_id", "category"]].assign(text="").to_csv(f"{pred_dir}/data/pages.csv", index=False)
!cp data/metadata.csv {pred_dir}/data/ && ln -sfn {work}/data/images {pred_dir}/images

!python churro_transcribe_batch.py \
    --model_id stanford-oval/churro-3B --adapter_dir output/{run_name} --dtype auto \
    --input_dir {pred_dir} --output_dir {pred_dir}/out --pages_csv pages.csv --batch_size 2

preds = pd.read_csv(f"{pred_dir}/out/predictions.csv", keep_default_na=False)
test_preds = test_df[["page_id"]].merge(preds, on="page_id", how="left").fillna({"text": ""})
assert len(test_preds) == len(test_df) and test_preds.text.str.len().gt(0).all(), "every test page needs a prediction"
test_preds.to_csv("test_predictions_churro_lora.csv", index=False)
print("wrote test_predictions_churro_lora.csv,", len(test_preds), "rows; download it from the file browser")
print(test_preds.head(3).to_string())

## What to take away

- Fine-tuning is training that starts from a trained model, so hundreds of examples do what millions would from scratch.
- The GPU's memory decides the method. A full fine-tune of a 3B model needs about 60 GB. LoRA trains under 1% of the parameters and needs about 12 GB. QLoRA stores the frozen weights in 4-bit and needs about 6 GB, which is what fits a free T4.
- Measure before you train. Tokens per page set the time, the adapter rank sets the trainable parameters, and both come from the model's own processor and config file.
- Smoke-test on a slice first, checkpoint every epoch, and compare the model before and after on the same held-out pages with the task's own metric. The holdout curve, not a rule of thumb, says how many passes were worth it.
- The same run on a rented cloud GPU costs about a dollar and finishes in half the time, and on UW-Madison's CHTC it is free and faster still. The [lesson](https://carpentries-incubator.github.io/ML_with_AWS_SageMaker/08_Finetuning-BadgerScribe.html) this notebook comes from runs it as a SageMaker job with the cost estimated before the launch and read back after it, and the folder beside this one has the CHTC version.